# População de referência — IBGE / 10ª CRS

Integra a tabela 9514 da API SIDRA (Censo Demográfico 2022) à arquitetura Medallion.

| Camada | Tabela | Granularidade |
| --- | --- | --- |
| Bronze | `dim_populacao_ibge_2022_raw` | Município × sexo × categoria de idade × forma de declaração |
| Silver | `dim_populacao_ibge_2022` | Município × faixa quinquenal feminina de 25 a 64 anos |
| Gold | `dim_populacao_municipios` | Município |

O Censo 2022 é o denominador fixo do período analítico de 2020 a 2025. A razão de exames por 100 mulheres não representa mulheres distintas rastreadas.

## 1. Configuração e dependências

Executar este notebook antes do `03_gold_analytics_siscan`. A extração, as transformações e as validações das três camadas são realizadas antes de qualquer gravação. A regressão fica habilitada por padrão.

A API SIDRA utiliza códigos municipais de sete dígitos; os seis primeiros correspondem ao identificador adotado pelo SISCAN. As categorias 93088–93095 correspondem às oito faixas quinquenais de 25 a 64 anos.

In [0]:
import re
import time
import unicodedata
from collections.abc import Sequence
from typing import Any

import pandas as pd
import requests
from pyspark.sql import DataFrame
from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType

CATALOG = "`tcc-analise-siscan-10crs`"
ANO = 2022
BASE_SIDRA = "https://apisidra.ibge.gov.br/values"
TABELA_SIDRA = 9514
VARIAVEL_POPULACAO = 93
TENTATIVAS = 3
TIMEOUT_SECONDS = 120
VALIDAR_REGRESSAO = True

MUNICIPIOS_10CRS = {
    "430040": "Alegrete",
    "430187": "Barra do Quaraí",
    "431060": "Itaqui",
    "431171": "Maçambara",
    "431175": "Manoel Viana",
    "431530": "Quaraí",
    "431640": "Rosário do Sul",
    "431697": "Santa Margarida do Sul",
    "431710": "Sant'Ana do Livramento",
    "431830": "São Gabriel",
    "432240": "Uruguaiana",
}
FAIXAS_25_64 = {
    str(code): (age, age + 4)
    for code, age in zip(range(93088, 93096), range(25, 65, 5))
}
SEXO_FEMININO = "Mulheres"
DECLARACAO_TOTAL = "Total"
CODIGO_DECLARACAO_TOTAL = "113635"
CODIGO_IDADE_TOTAL = "100362"

BRONZE_COLS = (
    "nivel_territorial_codigo", "nivel_territorial",
    "unidade_de_medida_codigo", "unidade_de_medida", "valor",
    "municipio_codigo", "municipio", "variavel_codigo", "variavel",
    "ano_codigo", "ano", "sexo_codigo", "sexo", "idade_codigo", "idade",
    "forma_de_declaracao_da_idade_codigo", "forma_de_declaracao_da_idade", "cd_ibge6",
)
BRONZE_SCHEMA = StructType([
    *[StructField(name, StringType(), True) for name in BRONZE_COLS],
    StructField("dt_ingestao", TimestampType(), True),
])
SILVER_SCHEMA = StructType([
    *[StructField(name, StringType(), True) for name in ("cd_ibge", "municipio", "cd_faixa_ibge", "faixa_etaria")],
    *[StructField(name, LongType(), True) for name in ("faixa_inicio", "faixa_fim", "populacao_mulheres", "ano")],
    StructField("dt_ingestao", TimestampType(), True),
])
GOLD_SCHEMA = StructType([
    StructField("cd_ibge", StringType(), True),
    StructField("municipio", StringType(), True),
    *[StructField(name, LongType(), True) for name in ("populacao_feminina", "ano", "mulheres_25_64_anos")],
])

bronze_table = f"{CATALOG}.bronze.dim_populacao_ibge_{ANO}_raw"
silver_table = f"{CATALOG}.silver.dim_populacao_ibge_{ANO}"
gold_table = f"{CATALOG}.gold.dim_populacao_municipios"


def exigir(condicao: bool, mensagem: str) -> None:
    if not condicao:
        raise ValueError(mensagem)


def sidra_get(path: str) -> list[dict[str, Any]]:
    with requests.Session() as session:
        for tentativa in range(1, TENTATIVAS + 1):
            try:
                response = session.get(f"{BASE_SIDRA}{path}", timeout=TIMEOUT_SECONDS)
                response.raise_for_status()
                return response.json()
            except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as error:
                status = (
                    error.response.status_code
                    if isinstance(error, requests.HTTPError) and error.response is not None
                    else None
                )
                retryable = status is None or status in (408, 429, 500, 502, 503, 504)
                if not retryable or tentativa == TENTATIVAS:
                    raise
                print(f"SIDRA: tentativa {tentativa}/{TENTATIVAS} falhou; nova tentativa.")
                time.sleep(3 * tentativa)
    raise RuntimeError("Consulta SIDRA encerrada sem resposta")


def parse_sidra(data: list[dict[str, Any]]) -> pd.DataFrame:
    exigir(isinstance(data, list) and len(data) > 1, "SIDRA retornou uma resposta vazia ou inválida")
    headers = data[0]
    exigir(isinstance(headers, dict), "Cabeçalho SIDRA inválido")
    exigir(len(set(headers.values())) == len(headers), "Cabeçalho SIDRA contém nomes duplicados")
    exigir(
        all(isinstance(row, dict) and set(row) == set(headers) for row in data[1:]),
        "Resposta SIDRA contém registros incompatíveis com o cabeçalho",
    )
    return pd.DataFrame(data[1:], columns=list(headers)).rename(columns=headers)


def filtrar_10crs(df: pd.DataFrame) -> pd.DataFrame:
    exigir("Município (Código)" in df, "Código municipal ausente na resposta SIDRA")
    result = df.assign(cd_ibge6=df["Município (Código)"].str[:6])
    return result.loc[result["cd_ibge6"].isin(MUNICIPIOS_10CRS)].copy()


def slug(nome: str) -> str:
    ascii_name = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    return re.sub(r"[^0-9a-zA-Z]+", "_", ascii_name).strip("_").lower()


def validar_chave(df: pd.DataFrame, chave: Sequence[str], nome: str) -> None:
    exigir(not df.empty, f"{nome}: tabela vazia")
    exigir(not df[list(chave)].isna().any().any(), f"{nome}: chave com valores nulos")
    exigir(not df.duplicated(subset=list(chave)).any(), f"{nome}: chave duplicada")


def validar_municipios(df: pd.DataFrame, coluna: str, nome: str) -> None:
    observados = set(df[coluna])
    exigir(
        observados == set(MUNICIPIOS_10CRS),
        f"{nome}: municípios ausentes={sorted(set(MUNICIPIOS_10CRS) - observados)}; "
        f"inesperados={sorted(observados - set(MUNICIPIOS_10CRS))}",
    )


def populacao_inteira(valores: pd.Series) -> pd.Series:
    numeros = pd.to_numeric(valores, errors="raise")
    exigir(
        not numeros.isna().any() and bool((numeros >= 0).all()) and bool((numeros % 1 == 0).all()),
        "População deve conter apenas inteiros não negativos",
    )
    return numeros.astype("int64")


def validar_regressao(df: DataFrame, tabela: str) -> None:
    if not VALIDAR_REGRESSAO or not spark.catalog.tableExists(tabela):
        return

    atual = spark.table(tabela)
    exigir(set(df.columns) == set(atual.columns), f"{tabela}: contrato de colunas alterado")
    tipos_atuais = {field.name: field.dataType for field in atual.schema.fields}
    exigir(
        all(field.dataType == tipos_atuais[field.name] for field in df.schema.fields),
        f"{tabela}: contrato de tipos alterado",
    )
    campos = [name for name in df.columns if name != "dt_ingestao"]
    anterior = atual.select(*campos)
    proposto = df.select(*campos)
    removidos = anterior.exceptAll(proposto).count()
    novos = proposto.exceptAll(anterior).count()
    exigir(
        removidos == 0 and novos == 0,
        f"{tabela}: regressão — {removidos} registros removidos/alterados; {novos} novos/alterados",
    )


def gravar_tabela(df: DataFrame, tabela: str) -> None:
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela)
    )

## 2. Bronze — extração bruta

Consulta o Censo 2022 para homens e mulheres, todas as categorias de idade e os 11 municípios. Preserva os valores da API; apenas normaliza os nomes dos campos e adiciona `cd_ibge6` e `dt_ingestao`. Não agrega categorias de idade sobrepostas.

In [0]:
def extrair_populacao() -> pd.DataFrame:
    # O período é fixado em 2022 para garantir a reprodutibilidade do TCC.
    path = f"/t/{TABELA_SIDRA}/n6/in n3 43/v/{VARIAVEL_POPULACAO}/p/{ANO}"
    try:
        return filtrar_10crs(parse_sidra(sidra_get(f"{path}/c2/4,5/c287/all")))
    except requests.HTTPError as error:
        if error.response is None or error.response.status_code not in (400, 413, 414, 422):
            raise
        print("SIDRA: consulta conjunta recusada; consulta separada por sexo.")
        frames = [
            filtrar_10crs(parse_sidra(sidra_get(f"{path}/c2/{sexo}/c287/all")))
            for sexo in ("4", "5")
        ]
        return pd.concat(frames, ignore_index=True)


df_bronze = extrair_populacao()
df_bronze.columns = [slug(name) for name in df_bronze.columns]
exigir(df_bronze.columns.is_unique, "Bronze: nomes de coluna duplicados após normalização")
campos_obrigatorios = {
    "cd_ibge6", "municipio_codigo", "municipio", "ano", "sexo", "sexo_codigo",
    "idade", "idade_codigo", "forma_de_declaracao_da_idade",
    "forma_de_declaracao_da_idade_codigo", "valor",
}
exigir(campos_obrigatorios.issubset(df_bronze.columns), "Bronze: estrutura inesperada da API SIDRA")
exigir(set(df_bronze.columns) == set(BRONZE_COLS), "Bronze: contrato de colunas da API alterado")
df_bronze = df_bronze.loc[:, list(BRONZE_COLS)].copy()
validar_municipios(df_bronze, "cd_ibge6", "Bronze")
exigir(set(df_bronze["ano"].astype(str)) == {str(ANO)}, "Bronze: ano diferente de 2022")
exigir(set(df_bronze["sexo_codigo"].astype(str)) == {"4", "5"}, "Bronze: sexo ausente ou inesperado")
validar_chave(
    df_bronze,
    ["municipio_codigo", "ano", "sexo_codigo", "idade_codigo", "forma_de_declaracao_da_idade_codigo"],
    "Bronze",
)

categorias = set(zip(df_bronze["idade_codigo"], df_bronze["forma_de_declaracao_da_idade_codigo"]))
pares_esperados = {
    (municipio, sexo, idade, declaracao)
    for municipio in MUNICIPIOS_10CRS
    for sexo in ("4", "5")
    for idade, declaracao in categorias
}
pares_observados = set(zip(
    df_bronze["cd_ibge6"], df_bronze["sexo_codigo"],
    df_bronze["idade_codigo"], df_bronze["forma_de_declaracao_da_idade_codigo"],
))
exigir(pares_observados == pares_esperados, "Bronze: grade de municípios, sexos e idades incompleta")

df_bronze["dt_ingestao"] = pd.Timestamp.now(tz="America/Sao_Paulo")
bronze_df = spark.createDataFrame(df_bronze, schema=BRONZE_SCHEMA)
print(f"Bronze: {len(df_bronze)} registros; {df_bronze['idade_codigo'].nunique()} categorias de idade.")
display(df_bronze.head(20))

## 3. Silver — população-alvo

Seleciona mulheres de 25 a 64 anos e a forma de declaração Total. Mantém as oito faixas quinquenais por município, com população inteira e ano de referência.

In [0]:
df_silver = df_bronze.loc[
    (df_bronze["sexo"] == SEXO_FEMININO)
    & (df_bronze["sexo_codigo"] == "5")
    & (df_bronze["forma_de_declaracao_da_idade"] == DECLARACAO_TOTAL)
    & (df_bronze["forma_de_declaracao_da_idade_codigo"] == CODIGO_DECLARACAO_TOTAL)
    & (df_bronze["idade_codigo"].isin(FAIXAS_25_64))
].copy()

df_silver = df_silver.assign(
    cd_ibge=df_silver["cd_ibge6"],
    municipio=df_silver["municipio"].str.replace(" - RS", "", regex=False).str.strip(),
    cd_faixa_ibge=df_silver["idade_codigo"],
    faixa_etaria=df_silver["idade"],
    faixa_inicio=df_silver["idade"].str.extract(r"^(\d+) a \d+ anos$")[0].astype("int64"),
    faixa_fim=df_silver["idade"].str.extract(r"^\d+ a (\d+) anos$")[0].astype("int64"),
    populacao_mulheres=populacao_inteira(df_silver["valor"]),
    ano=df_silver["ano"].astype("int64"),
)[[
    "cd_ibge", "municipio", "cd_faixa_ibge", "faixa_etaria", "faixa_inicio",
    "faixa_fim", "populacao_mulheres", "ano", "dt_ingestao",
]]

validar_chave(df_silver, ["cd_ibge", "cd_faixa_ibge", "ano"], "Silver")
validar_municipios(df_silver, "cd_ibge", "Silver")
grade_esperada = {
    (municipio, faixa) for municipio in MUNICIPIOS_10CRS for faixa in FAIXAS_25_64
}
grade_observada = set(zip(df_silver["cd_ibge"], df_silver["cd_faixa_ibge"]))
exigir(grade_observada == grade_esperada, "Silver: grade município × faixa etária incompleta")
exigir(
    all(
        FAIXAS_25_64[row.cd_faixa_ibge] == (row.faixa_inicio, row.faixa_fim)
        for row in df_silver.itertuples(index=False)
    ),
    "Silver: códigos SIDRA e limites etários divergentes",
)
exigir(not df_silver.isna().any().any(), "Silver: campos obrigatórios nulos")

silver_df = spark.createDataFrame(df_silver, schema=SILVER_SCHEMA)
print(f"Silver: {len(df_silver)} registros.")
display(df_silver.sort_values(["municipio", "faixa_inicio"]))

## 4. Gold — dimensão municipal e persistência

População feminina total: linha de idade Total da Bronze. População-alvo: soma das oito faixas da Silver. A dimensão mantém os nomes e tipos consumidos pela Gold analítica.

A carga valida as três tabelas antes do overwrite. Cada gravação Delta é transacional por tabela, não em conjunto. Desative `VALIDAR_REGRESSAO` somente após aprovar uma alteração intencional da fonte.

In [0]:
df_mulheres_25_64 = (
    df_silver.groupby(["cd_ibge", "municipio", "ano"], as_index=False)["populacao_mulheres"]
    .sum()
    .rename(columns={"populacao_mulheres": "mulheres_25_64_anos"})
)

# A idade Total não deve ser somada às categorias quinquenais ou às idades simples.
df_fem_total = (
    df_bronze.loc[
        (df_bronze["sexo"] == SEXO_FEMININO)
        & (df_bronze["sexo_codigo"] == "5")
        & (df_bronze["idade"] == "Total")
        & (df_bronze["idade_codigo"] == CODIGO_IDADE_TOTAL)
        & (df_bronze["forma_de_declaracao_da_idade"] == DECLARACAO_TOTAL)
        & (df_bronze["forma_de_declaracao_da_idade_codigo"] == CODIGO_DECLARACAO_TOTAL)
    ]
    .assign(populacao_feminina=lambda df: populacao_inteira(df["valor"]))
    .rename(columns={"cd_ibge6": "cd_ibge"})
    [["cd_ibge", "populacao_feminina"]]
)
validar_chave(df_fem_total, ["cd_ibge"], "População feminina total")
validar_municipios(df_fem_total, "cd_ibge", "População feminina total")

df_populacao_municipios = df_fem_total.merge(
    df_mulheres_25_64,
    on="cd_ibge",
    how="inner",
    validate="one_to_one",
)[["cd_ibge", "municipio", "populacao_feminina", "ano", "mulheres_25_64_anos"]]

validar_chave(df_populacao_municipios, ["cd_ibge"], "Gold")
validar_municipios(df_populacao_municipios, "cd_ibge", "Gold")
exigir(not df_populacao_municipios.isna().any().any(), "Gold: campos obrigatórios nulos")
exigir(
    bool((df_populacao_municipios["mulheres_25_64_anos"] > 0).all())
    and bool((df_populacao_municipios["mulheres_25_64_anos"] <= df_populacao_municipios["populacao_feminina"]).all()),
    "Gold: denominador inválido",
)
exigir(
    int(df_populacao_municipios["mulheres_25_64_anos"].sum()) == int(df_silver["populacao_mulheres"].sum()),
    "Gold: total da população-alvo diverge da Silver",
)
gold_df = spark.createDataFrame(df_populacao_municipios, schema=GOLD_SCHEMA)

tabelas = [
    (bronze_table, bronze_df, len(df_bronze)),
    (silver_table, silver_df, len(df_silver)),
    (gold_table, gold_df, len(df_populacao_municipios)),
]
for schema in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

for tabela, df, linhas in tabelas:
    exigir(df.count() == linhas, f"{tabela}: contagem divergente após conversão para Spark")
    validar_regressao(df, tabela)
    print(f"{tabela}: {linhas} registros validados.")

for tabela, df, linhas in tabelas:
    gravar_tabela(df, tabela)
    exigir(spark.table(tabela).count() == linhas, f"{tabela}: contagem publicada divergente")
    print(f"{tabela}: {linhas} registros gravados.")

print(f"População feminina regional: {df_populacao_municipios['populacao_feminina'].sum():,}")
print(f"Mulheres de 25 a 64 anos: {df_populacao_municipios['mulheres_25_64_anos'].sum():,}")
display(df_populacao_municipios.sort_values("populacao_feminina", ascending=False))